Чтобы посчитать COMET-22 пришлось повозиться с зависимостями, рабочий вариант который я получил:

In [1]:
!pip install -q uv
!uv python install 3.11
!uv venv /kaggle/working/comet_env --python 3.11

Installed Python 3.11.18 in 1.20s
 + cpython-3.11.18-linux-x86_64-gnu (python3.11)
Using CPython 3.11.18
Creating virtual environment at: comet_env
Activate with: source comet_env/bin/activate


In [5]:
!uv pip install --python /kaggle/working/comet_env/bin/python \
    "unbabel-comet==2.2.7" \
    "transformers==4.44.2" \
    "numpy==1.26.4" \
    "pandas==2.2.3" \
    "setuptools==80.9.0"

Using Python 3.11.18 environment at: comet_env
Resolved 70 packages in 137ms                                        
Prepared 1 package in 198ms                                              
Uninstalled 1 package in 10ms
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 48ms                                 
 - setuptools==84.2.1
 + setuptools==80.9.0


In [9]:
%%writefile /kaggle/working/score_comet.py
import json
import pandas as pd
from comet import download_model, load_from_checkpoint

PATH = "/kaggle/input/datasets/ivansher/entropy-valley-results/ev_400_results.jsonl"

with open(PATH, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

results = []

for method in ["ratio", "ev", "oracle"]:
    data = [
        {
            "src": row["source"],
            "mt": row["translations"][method],
            "ref": row["reference"],
        }
        for row in rows
    ]

    output = model.predict(data, batch_size=8, gpus=1)

    print(method, output.system_score, flush=True)

    results.append({
        "Method": method,
        "COMET-22": output.system_score,
    })

pd.DataFrame(results).to_csv(
    "/kaggle/working/ev_400_comet.csv",
    index=False,
)

print("Saved")

Overwriting /kaggle/working/score_comet.py


In [10]:
!/kaggle/working/comet_env/bin/python /kaggle/working/score_comet.py

/kaggle/working/comet_env/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files:   0%|                                  | 0/5 [00:00<?, ?it/s]
LICENSE: 9.69kB [00:00, 22.0MB/s]
Fetching 5 files:  20%|█████▏                    | 1/5 [00:00<00:00,  7.36it/s]
README.md: 3.40kB [00:00, 20.8MB/s]

checkpoints/model.ckpt:   0%|                      | 0.00/2.32G [00:00<?, ?B/s]

.gitattributes: 1.48kB [00:00, 8.93MB/s]A


hparams.yaml: 100%|███████████████████████████| 567/567 [00:00<00:00, 5.69MB/s]

checkpoints/model.ckpt:   0%|                      | 0.00/2.32G [00:00<?, ?B/s]
checkpoints/model.ckpt:   0%|             | 194k/2.32G [00:00<1:39:59, 387kB/s]
c